# OralPath Colab Bootstrap

Use this notebook from VS Code with the official Google Colab extension. The notebook assigns a Colab runtime, prepares Drive artifact folders, clones/updates the repo, and launches repo-owned training scripts.

Do not paste one-shot training scripts here. Keep reusable training logic in `model/training/`.

## 1. Check GPU

In [ ]:
!nvidia-smi || true

import torch
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))

## 2. Mount Google Drive

Drive stores datasets, checkpoints, run logs, and exports. Source code stays in git and is cloned into `/content/oralpath`.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path
base = Path('/content/drive/MyDrive/oralpath')
for name in ['data', 'runs', 'checkpoints', 'exports']:
    (base / name).mkdir(parents=True, exist_ok=True)
print('Drive workspace:', base)

## 3. Clone Or Update Repo

Set `REPO_URL` after the project is pushed to GitHub. For private repos, use a temporary access token or Colab's GitHub auth flow. Do not hardcode secrets in the notebook.

In [ ]:
from pathlib import Path

REPO_URL = 'https://github.com/ORION2809/OSCC.git'
repo_dir = Path('/content/oralpath')

if not REPO_URL:
    raise ValueError('Set REPO_URL before cloning the project into Colab.')

if repo_dir.exists():
    %cd /content/oralpath
    !git pull --ff-only
else:
    %cd /content
    !git clone {REPO_URL} oralpath
    %cd /content/oralpath

print('Repo ready at', repo_dir)

## 4. Install Dependencies

In [ ]:
%cd /content/oralpath
!pwd
!ls -la
from pathlib import Path
if not Path('requirements.txt').exists():
    raise FileNotFoundError('requirements.txt not found in /content/oralpath. Check REPO_URL, branch, and whether this local repo has been pushed to GitHub.')
!python -m pip install --upgrade pip
!pip install -r requirements.txt

## 5. Runtime And Dataset Dry Runs

## 5A. Link Drive Datasets

Before this cell can pass, Drive must contain only the dataset folders:

- `/content/drive/MyDrive/oralpath/data/kaggle_oscc`
- `/content/drive/MyDrive/oralpath/data/orchid`

These are not stored in GitHub.

## 5B. No-Drive Alternative: Temporary Colab Data

Use this if Google Drive does not have enough space. It downloads data into `/content/oralpath/model/data/...`. The data disappears when the runtime is recycled, but it does not consume Drive quota.

Stage 1 is the recommended first run. Stage 2 ORCHID is much larger and should be attempted only if the runtime has enough free disk.

In [ ]:
%cd /content/oralpath
# Stage 1 only, no Drive storage. Requires Kaggle API credentials.
!python scripts/setup_colab_ephemeral_data.py --stage stage1
!python scripts/verify_datasets.py

In [ ]:
%cd /content/oralpath
!python scripts/setup_colab_datasets.py --drive-root /content/drive/MyDrive/oralpath/data
!python scripts/verify_datasets.py

In [ ]:
%cd /content/oralpath
!python scripts/colab_runtime_check.py
!python model/data/preprocessing/dataset_loader.py
!python model/training/stage1_detection/train.py --config model/training/stage1_detection/config.yaml --dry-run
!python model/training/stage2_grading/train.py --config model/training/stage2_grading/config.yaml --dry-run

## 6. Stage 1 Smoke Test

Run two batches first. If this fails, fix the repo script or dataset paths before launching the full experiment.

In [ ]:
%cd /content/oralpath
!python model/training/stage1_detection/train.py --config model/training/stage1_detection/config.yaml --max-batches 2

## 7. Stage 1 Full Training

Run this only after the smoke test completes.

In [ ]:
%cd /content/oralpath
!python model/training/stage1_detection/train.py --config model/training/stage1_detection/config.yaml

## 8. Stage 2 Smoke Test

Run after ORCHID is available under `model/data/processed/orchid` or after the manifest points to the Colab Drive ORCHID path.

In [ ]:
%cd /content/oralpath
!python model/training/stage2_grading/train.py --config model/training/stage2_grading/config.yaml --max-batches 2

## 9. Stage 2 Full Training

Run this only after the Stage 2 smoke test completes.

In [ ]:
%cd /content/oralpath
!python model/training/stage2_grading/train.py --config model/training/stage2_grading/config.yaml